In [ ]:
import json
import os
import random
import time
from datetime import datetime, timedelta

from kafka import KafkaProducer
from kafka.errors import KafkaError, NoBrokersAvailable


# =========================
# KONFIGURACJA KAFKA
# =========================

KAFKA_BOOTSTRAP_SERVERS = "broker:9092"

ORDERS_TOPIC = "restaurant_orders"
PANTRY_TOPIC = "pantry_events"

CONTROL_STATE_PATH = "/home/jovyan/notebooks/control_state.json"

DEFAULT_CONTROL_STATE = {
    "current_staff": 3,
    "manual_restock": {}
}

# =========================
# KONFIGURACJA SPIŻARNI / MAGAZYNU
# =========================

# Stan początkowy spiżarni.
# Wartości są celowo niewielkie, żeby podczas demo było widać,
# że składniki się kończą i później są uzupełniane.

INITIAL_PANTRY = {
    "pizza_dough": 20,
    "tomato_sauce": 35,
    "cheese": 45,
    "pepperoni": 18,
    "bun": 25,
    "beef_patty": 22,
    "lettuce": 25,
    "pasta": 25,
    "bacon": 18,
    "fries": 30,
    "chicken": 20,
    "caesar_sauce": 18,
    "soup_base": 20
}

# Maksymalna pojemność spiżarni.
# Dzięki temu po uzupełnieniu wartości nie rosną w nieskończoność.

PANTRY_CAPACITY = {
    "pizza_dough": 50,
    "tomato_sauce": 80,
    "cheese": 100,
    "pepperoni": 50,
    "bun": 60,
    "beef_patty": 60,
    "lettuce": 60,
    "pasta": 60,
    "bacon": 50,
    "fries": 80,
    "chicken": 50,
    "caesar_sauce": 50,
    "soup_base": 50
}

# Co ile sekund spiżarnia jest uzupełniana.
# Na potrzeby demo: 120 sekund = 2 minuty.
# W realnym opisie projektu można powiedzieć, że to skrócony czas demonstracyjny.

PANTRY_RESTOCK_INTERVAL_SECONDS = 120

# Plan uzupełnienia spiżarni.
# Co każde uzupełnienie dodawane są takie ilości składników.

PANTRY_RESTOCK_PLAN = {
    "pizza_dough": 10,
    "tomato_sauce": 15,
    "cheese": 20,
    "pepperoni": 10,
    "bun": 12,
    "beef_patty": 12,
    "lettuce": 10,
    "pasta": 10,
    "bacon": 8,
    "fries": 15,
    "chicken": 10,
    "caesar_sauce": 8,
    "soup_base": 8
}


def load_control_state():
    """
    Wczytuje decyzje menadżera z pliku control_state.json.
    Jeśli plik nie istnieje, tworzy domyślny stan.
    """
    if not os.path.exists(CONTROL_STATE_PATH):
        save_control_state(DEFAULT_CONTROL_STATE)
        return DEFAULT_CONTROL_STATE.copy()

    try:
        with open(CONTROL_STATE_PATH, "r", encoding="utf-8") as file:
            control_state = json.load(file)

        if "current_staff" not in control_state:
            control_state["current_staff"] = 3

        if "manual_restock" not in control_state:
            control_state["manual_restock"] = {}

        return control_state

    except Exception:
        return DEFAULT_CONTROL_STATE.copy()


def save_control_state(control_state):
    """
    Zapisuje stan kontroli do pliku JSON.
    """
    with open(CONTROL_STATE_PATH, "w", encoding="utf-8") as file:
        json.dump(control_state, file, ensure_ascii=False, indent=2)


def apply_manual_restock(pantry, manual_restock):
    """
    Dodaje składniki ręcznie wskazane przez menadżera.
    Nie przekracza maksymalnej pojemności spiżarni.
    """
    added_items = {}

    for ingredient, amount in manual_restock.items():
        if ingredient not in pantry:
            continue

        amount = int(amount)

        if amount <= 0:
            continue

        current_amount = pantry.get(ingredient, 0)
        capacity = PANTRY_CAPACITY.get(ingredient, current_amount + amount)

        new_amount = min(current_amount + amount, capacity)
        added_amount = new_amount - current_amount

        pantry[ingredient] = new_amount
        added_items[ingredient] = added_amount

    return added_items


def create_manual_restock_event(pantry, restocked_items):
    """
    Tworzy event ręcznego uzupełnienia spiżarni przez menadżera.
    """
    event_time = datetime.now()

    event = {
        "event_type": "manual_pantry_restock",
        "timestamp": event_time.isoformat(timespec="seconds"),
        "restocked_items": restocked_items,
        "pantry_after_restock": pantry.copy(),
        "low_stock_after_restock": get_low_stock_ingredients(pantry)
    }

    return event



# =========================
# MENU RESTAURACJI
# =========================
# Każde danie ma:
# - cenę,
# - oczekiwany czas przygotowania w sekundach,
# - listę składników potrzebnych do przygotowania jednej sztuki.

MENU = [
    {
        "dish_name": "Pizza Margherita",
        "dish_category": "Pizza",
        "price": 29.90,
        "expected_preparation_seconds": 60,
        "ingredients": {
            "pizza_dough": 1,
            "tomato_sauce": 1,
            "cheese": 2
        }
    },
    {
        "dish_name": "Pizza Pepperoni",
        "dish_category": "Pizza",
        "price": 34.90,
        "expected_preparation_seconds": 70,
        "ingredients": {
            "pizza_dough": 1,
            "tomato_sauce": 1,
            "cheese": 2,
            "pepperoni": 1
        }
    },
    {
        "dish_name": "Pizza Capricciosa",
        "dish_category": "Pizza",
        "price": 36.90,
        "expected_preparation_seconds": 75,
        "ingredients": {
            "pizza_dough": 1,
            "tomato_sauce": 1,
            "cheese": 2,
            "pepperoni": 1
        }
    },
    {
        "dish_name": "Burger klasyczny",
        "dish_category": "Burger",
        "price": 31.90,
        "expected_preparation_seconds": 40,
        "ingredients": {
            "bun": 1,
            "beef_patty": 1,
            "cheese": 1,
            "lettuce": 1
        }
    },
    {
        "dish_name": "Burger BBQ",
        "dish_category": "Burger",
        "price": 36.90,
        "expected_preparation_seconds": 45,
        "ingredients": {
            "bun": 1,
            "beef_patty": 1,
            "cheese": 1,
            "bacon": 1,
            "lettuce": 1
        }
    },
    {
        "dish_name": "Makaron Carbonara",
        "dish_category": "Makaron",
        "price": 32.90,
        "expected_preparation_seconds": 50,
        "ingredients": {
            "pasta": 1,
            "bacon": 1,
            "cheese": 1
        }
    },
    {
        "dish_name": "Makaron Bolognese",
        "dish_category": "Makaron",
        "price": 34.90,
        "expected_preparation_seconds": 55,
        "ingredients": {
            "pasta": 1,
            "tomato_sauce": 1,
            "cheese": 1
        }
    },
    {
        "dish_name": "Sałatka Cezar",
        "dish_category": "Sałatka",
        "price": 27.90,
        "expected_preparation_seconds": 25,
        "ingredients": {
            "lettuce": 2,
            "chicken": 1,
            "caesar_sauce": 1,
            "cheese": 1
        }
    },
    {
        "dish_name": "Zupa dnia",
        "dish_category": "Zupa",
        "price": 18.90,
        "expected_preparation_seconds": 20,
        "ingredients": {
            "soup_base": 1
        }
    },
    {
        "dish_name": "Frytki",
        "dish_category": "Dodatek",
        "price": 12.90,
        "expected_preparation_seconds": 15,
        "ingredients": {
            "fries": 1
        }
    }
]


ORDER_CHANNELS = ["app", "website", "phone", "restaurant"]


# =========================
# POŁĄCZENIE Z KAFKA
# =========================

def create_producer():
    """
    Tworzy producenta Kafka.
    Producer wysyła:
    - zamówienia do restaurant_orders,
    - zdarzenia spiżarni do pantry_events.
    """
    try:
        print(f"Próba połączenia z Kafka: {KAFKA_BOOTSTRAP_SERVERS}")

        producer = KafkaProducer(
            bootstrap_servers=[KAFKA_BOOTSTRAP_SERVERS],
            value_serializer=lambda value: json.dumps(
                value,
                ensure_ascii=False
            ).encode("utf-8"),
            api_version=(2, 8, 0),
            request_timeout_ms=30000,
            max_block_ms=30000,
            retries=3
        )

        print("Połączono z Kafka.")
        return producer

    except NoBrokersAvailable:
        print("Nie znaleziono brokera Kafka.")
        print("Sprawdź, czy Kafka działa oraz czy adres bootstrap servera jest poprawny.")
        raise

    except Exception as e:
        print(f"Wystąpił błąd podczas tworzenia producenta Kafka: {e}")
        raise



# =========================
# LOGIKA POPYTU I PROMOCJI
# =========================

def is_promotion_active():
    """
    Promocja aktywna codziennie w godzinach 18:00-20:59.
    """
    current_hour = datetime.now().hour
    return 18 <= current_hour <= 20


def get_demand_multiplier():
    """
    Określa natężenie ruchu.
    Większy mnożnik oznacza częstsze generowanie zamówień.
    """
    current_hour = datetime.now().hour
    multiplier = 1.0

    if 12 <= current_hour <= 14:
        multiplier *= 1.8

    elif 18 <= current_hour <= 21:
        multiplier *= 2.2

    elif current_hour >= 22 or current_hour <= 6:
        multiplier *= 0.4

    if is_promotion_active():
        multiplier *= 1.3

    return multiplier


def get_demand_level(demand_multiplier):
    """
    Zamienia mnożnik popytu na prostą kategorię biznesową.
    """
    if demand_multiplier < 0.8:
        return "low"
    elif demand_multiplier < 1.4:
        return "normal"
    elif demand_multiplier < 2.3:
        return "high"
    else:
        return "very_high"


def get_sleep_time():
    """
    Ustala, jak często pojawiają się zamówienia.
    Wersja wolniejsza, żeby kuchnia nie przeciążała się od razu.
    """
    demand_multiplier = get_demand_multiplier()

    base_sleep = random.uniform(6, 12)
    sleep_time = base_sleep / demand_multiplier

    return max(2.0, sleep_time)


# =========================
# LOGIKA SPIŻARNI
# =========================

def calculate_ingredients_used(dish, quantity):
    """
    Oblicza, ile składników zużywa dane zamówienie.
    """
    ingredients_used = {}

    for ingredient, amount_per_item in dish["ingredients"].items():
        ingredients_used[ingredient] = amount_per_item * quantity

    return ingredients_used


def check_ingredients_available(pantry, ingredients_used):
    """
    Sprawdza, czy w spiżarni są wszystkie potrzebne składniki.
    Zwraca:
    - True/False,
    - listę brakujących składników.
    """
    missing_ingredients = {}

    for ingredient, required_amount in ingredients_used.items():
        available_amount = pantry.get(ingredient, 0)

        if available_amount < required_amount:
            missing_ingredients[ingredient] = {
                "required": required_amount,
                "available": available_amount,
                "missing": required_amount - available_amount
            }

    is_available = len(missing_ingredients) == 0

    return is_available, missing_ingredients


def deduct_ingredients_from_pantry(pantry, ingredients_used):
    """
    Odejmuje zużyte składniki ze spiżarni.
    """
    for ingredient, used_amount in ingredients_used.items():
        pantry[ingredient] = pantry.get(ingredient, 0) - used_amount
        pantry[ingredient] = max(0, pantry[ingredient])


def get_low_stock_ingredients(pantry):
    """
    Wykrywa składniki, których poziom jest niski.
    Niski poziom oznacza mniej niż 20% maksymalnej pojemności.
    """
    low_stock = {}

    for ingredient, current_amount in pantry.items():
        capacity = PANTRY_CAPACITY.get(ingredient, current_amount)

        if capacity == 0:
            continue

        stock_ratio = current_amount / capacity

        if stock_ratio <= 0.2:
            low_stock[ingredient] = {
                "current_amount": current_amount,
                "capacity": capacity,
                "stock_ratio": round(stock_ratio, 2)
            }

    return low_stock


def restock_pantry(pantry):
    """
    Uzupełnia spiżarnię zgodnie z planem.
    Nie przekracza maksymalnej pojemności.
    Zwraca informację, ile faktycznie dodano każdego składnika.
    """
    restocked_items = {}

    for ingredient, planned_amount in PANTRY_RESTOCK_PLAN.items():
        current_amount = pantry.get(ingredient, 0)
        capacity = PANTRY_CAPACITY.get(ingredient, current_amount + planned_amount)

        new_amount = min(current_amount + planned_amount, capacity)
        added_amount = new_amount - current_amount

        pantry[ingredient] = new_amount
        restocked_items[ingredient] = added_amount

    return restocked_items


def create_pantry_restock_event(pantry, restocked_items):
    """
    Tworzy zdarzenie informujące o uzupełnieniu spiżarni.
    To zdarzenie idzie do osobnego topicu pantry_events.
    """
    event_time = datetime.now()

    event = {
        "event_type": "pantry_restock",
        "timestamp": event_time.isoformat(timespec="seconds"),
        "restock_interval_seconds": PANTRY_RESTOCK_INTERVAL_SECONDS,
        "restocked_items": restocked_items,
        "pantry_after_restock": pantry.copy(),
        "low_stock_after_restock": get_low_stock_ingredients(pantry)
    }

    return event


# =========================
# GENEROWANIE ZAMÓWIENIA
# =========================

def choose_dish():
    """
    Losuje danie z menu.
    Najpopularniejsze dania mają większą wagę.
    """
    weights = [
        0.14,
        0.16,
        0.10,
        0.14,
        0.12,
        0.10,
        0.08,
        0.07,
        0.04,
        0.05
    ]

    return random.choices(MENU, weights=weights, k=1)[0]


def generate_quantity():
    """
    Losuje liczbę sztuk w zamówieniu.
    """
    return random.choices(
        population=[1, 2, 3],
        weights=[0.65, 0.25, 0.10],
        k=1
    )[0]


def generate_actual_preparation_seconds(expected_seconds, current_staff):
    """
    Generuje faktyczny czas przygotowania.
    Uwzględnia popyt oraz liczbę pracowników.

    Im większy popyt, tym większe opóźnienia.
    Im więcej pracowników, tym krótszy czas przygotowania.
    """
    demand_multiplier = get_demand_multiplier()

    if demand_multiplier >= 2.0:
        delay = random.randint(0, 25)
    else:
        delay = random.randint(-5, 15)

    # 3 osoby = normalna obsada.
    # Każda dodatkowa osoba skraca czas przygotowania,
    # a każda brakująca osoba wydłuża czas przygotowania.
    staff_effect = (current_staff - 3) * 6

    actual_seconds = expected_seconds + delay - staff_effect

    return max(5, actual_seconds)


def generate_order(order_number, pantry, current_staff):
    """
    Generuje jedno zamówienie.
    Zamówienie zawiera:
    - dane sprzedażowe,
    - czasy przygotowania,
    - kontekst popytu,
    - zużyte składniki,
    - informację, czy składniki były dostępne,
    - stan spiżarni po zamówieniu.
    """
    dish = choose_dish()
    quantity = generate_quantity()

    created_at = datetime.now()

    expected_seconds = dish["expected_preparation_seconds"]
    actual_seconds = generate_actual_preparation_seconds(expected_seconds, current_staff)

    estimated_ready_at = created_at + timedelta(seconds=expected_seconds)
    actual_ready_at = created_at + timedelta(seconds=actual_seconds)

    is_delayed = actual_seconds > expected_seconds
    delay_seconds = max(0, actual_seconds - expected_seconds)

    demand_multiplier = get_demand_multiplier()
    demand_level = get_demand_level(demand_multiplier)

    promotion_active = is_promotion_active()

    ingredients_used = calculate_ingredients_used(dish, quantity)

    ingredients_available, missing_ingredients = check_ingredients_available(
        pantry,
        ingredients_used
    )

    if ingredients_available:
        deduct_ingredients_from_pantry(pantry, ingredients_used)
        order_status = "in_progress"
        cancellation_reason = None
    else:
        order_status = "cancelled_missing_ingredients"
        cancellation_reason = "Not enough ingredients in pantry"

        # Zamówienie anulowane nie powinno być aktywne na dashboardzie.
        # Dlatego actual_ready_at ustawiamy na czas utworzenia.
        actual_ready_at = created_at
        estimated_ready_at = created_at
        actual_seconds = 0
        delay_seconds = 0
        is_delayed = False

    low_stock_after_order = get_low_stock_ingredients(pantry)

    order = {
        "event_type": "order",

        "order_id": f"ORD_{order_number:06d}",

        "timestamp": created_at.isoformat(timespec="seconds"),
        "created_at": created_at.isoformat(timespec="seconds"),

        "dish_name": dish["dish_name"],
        "dish_category": dish["dish_category"],

        "quantity": quantity,
        "unit_price": dish["price"],
        "order_value": round(dish["price"] * quantity, 2),
        "order_channel": random.choice(ORDER_CHANNELS),

        "expected_preparation_seconds": expected_seconds,
        "actual_preparation_seconds": actual_seconds,

        "estimated_ready_at": estimated_ready_at.isoformat(timespec="seconds"),
        "actual_ready_at": actual_ready_at.isoformat(timespec="seconds"),

        "is_delayed": is_delayed,
        "delay_seconds": delay_seconds,

        "status": order_status,
        "cancellation_reason": cancellation_reason,

        "is_promotion": promotion_active,
        "demand_multiplier": round(demand_multiplier, 2),
        "demand_level": demand_level,
        "current_staff": current_staff,

        "ingredients_used": ingredients_used,
        "ingredients_available": ingredients_available,
        "missing_ingredients": missing_ingredients,

        "pantry_after_order": pantry.copy(),
        "low_stock_after_order": low_stock_after_order
    }

    return order


# =========================
# WYSYŁANIE DO KAFKA
# =========================

def send_message(producer, topic, message):
    """
    Wysyła wiadomość do wskazanego topicu Kafka.
    """
    try:
        future = producer.send(topic, value=message)
        record_metadata = future.get(timeout=30)

        return record_metadata

    except KafkaError as e:
        print("Błąd Kafka podczas wysyłania wiadomości:")
        print(e)
        raise

    except Exception as e:
        print("Inny błąd podczas wysyłania wiadomości:")
        print(e)
        raise


def log_order(order, metadata):
    """
    Wyświetla skrócony log zamówienia.
    """
    print(
        f"ZAMÓWIENIE | "
        f"{order['order_id']} | "
        f"{order['dish_name']} x{order['quantity']} | "
        f"status={order['status']} | "
        f"składniki_ok={order['ingredients_available']} | "
        f"popyt={order['demand_level']} | "
        f"promocja={order['is_promotion']} | "
        f"topic={metadata.topic}, "
        f"partition={metadata.partition}, "
        f"offset={metadata.offset}"
    )

    if not order["ingredients_available"]:
        print(f"  Brakujące składniki: {order['missing_ingredients']}")

    if order["low_stock_after_order"]:
        print(f"  Niski stan składników: {list(order['low_stock_after_order'].keys())}")


def log_pantry_event(event, metadata):
    """
    Wyświetla skrócony log uzupełnienia spiżarni.
    """
    added = {
        ingredient: amount
        for ingredient, amount in event["restocked_items"].items()
        if amount > 0
    }

    print(
        f"SPIŻARNIA UZUPEŁNIONA | "
        f"dodano={added} | "
        f"topic={metadata.topic}, "
        f"partition={metadata.partition}, "
        f"offset={metadata.offset}"
    )


def generate_actual_preparation_seconds(expected_seconds, current_staff):
    """
    Generuje faktyczny czas przygotowania.
    Uwzględnia popyt oraz liczbę pracowników.
    """
    demand_multiplier = get_demand_multiplier()

    if demand_multiplier >= 2.0:
        delay = random.randint(0, 25)
    else:
        delay = random.randint(-5, 15)

    # 3 osoby = normalna obsada.
    # Każda dodatkowa osoba skraca czas, a każda brakująca wydłuża.
    staff_effect = (current_staff - 3) * 6

    actual_seconds = expected_seconds + delay - staff_effect

    return max(5, actual_seconds)


# =========================
# GŁÓWNA PĘTLA PROGRAMU
# =========================
control_state = load_control_state()
current_staff = int(control_state.get("current_staff", 3))


def main():
    producer = create_producer()

    pantry = INITIAL_PANTRY.copy()
    order_number = 1
    last_restock_time = datetime.now()

    print("Producer uruchomiony.")
    print(f"Topic zamówień: {ORDERS_TOPIC}")
    print(f"Topic spiżarni: {PANTRY_TOPIC}")
    print(f"Spiżarnia będzie automatycznie uzupełniana co {PANTRY_RESTOCK_INTERVAL_SECONDS} sekund.")
    print("Producer czyta decyzje menadżera z pliku control_state.json.")
    print("Aby zatrzymać producer, użyj Kernel -> Interrupt albo Ctrl+C.")

    try:
        while True:
            now = datetime.now()

            # 1. Odczyt decyzji menadżera
            control_state = load_control_state()
            current_staff = int(control_state.get("current_staff", 3))
            manual_restock = control_state.get("manual_restock", {})

            if manual_restock:
                restocked_items = apply_manual_restock(pantry, manual_restock)

                if any(amount > 0 for amount in restocked_items.values()):
                    pantry_event = create_manual_restock_event(
                        pantry,
                        restocked_items
                    )

                    metadata = send_message(
                        producer,
                        PANTRY_TOPIC,
                        pantry_event
                    )

                    log_pantry_event(pantry_event, metadata)

                control_state["manual_restock"] = {}
                save_control_state(control_state)

            # 2. Automatyczne uzupełnienie spiżarni
            seconds_since_last_restock = (
                now - last_restock_time
            ).total_seconds()

            if seconds_since_last_restock >= PANTRY_RESTOCK_INTERVAL_SECONDS:
                restocked_items = restock_pantry(pantry)

                pantry_event = create_pantry_restock_event(
                    pantry,
                    restocked_items
                )

                metadata = send_message(
                    producer,
                    PANTRY_TOPIC,
                    pantry_event
                )

                log_pantry_event(pantry_event, metadata)

                last_restock_time = now

            # 3. Generowanie zamówienia
            order = generate_order(
                order_number,
                pantry,
                current_staff
            )

            metadata = send_message(
                producer,
                ORDERS_TOPIC,
                order
            )

            log_order(order, metadata)

            order_number += 1

            # 4. Pauza
            sleep_time = get_sleep_time()
            time.sleep(sleep_time)

    except KeyboardInterrupt:
        print("Producer został zatrzymany przez użytkownika.")

    finally:
        producer.flush()
        producer.close()
        print("Połączenie z Kafka zostało zamknięte.")


if __name__ == "__main__":
    main()


Próba połączenia z Kafka: broker:9092
Połączono z Kafka.
Producer uruchomiony.
Topic zamówień: restaurant_orders
Topic spiżarni: pantry_events
Spiżarnia będzie automatycznie uzupełniana co 120 sekund.
Producer czyta decyzje menadżera z pliku control_state.json.
Aby zatrzymać producer, użyj Kernel -> Interrupt albo Ctrl+C.
ZAMÓWIENIE | ORD_000001 | Burger klasyczny x2 | status=in_progress | składniki_ok=True | popyt=normal | promocja=False | topic=restaurant_orders, partition=0, offset=293
ZAMÓWIENIE | ORD_000002 | Pizza Pepperoni x1 | status=in_progress | składniki_ok=True | popyt=normal | promocja=False | topic=restaurant_orders, partition=0, offset=294
ZAMÓWIENIE | ORD_000003 | Burger klasyczny x1 | status=in_progress | składniki_ok=True | popyt=normal | promocja=False | topic=restaurant_orders, partition=0, offset=295
ZAMÓWIENIE | ORD_000004 | Pizza Margherita x3 | status=in_progress | składniki_ok=True | popyt=normal | promocja=False | topic=restaurant_orders, partition=0, offset=2